In [6]:
from pathlib import Path
import pandas as pd

# ---------------------------
# User-configurable paths
# ---------------------------
# manifest_csv = Path("table/runs_manifest_20260409_042243.csv")
# manifest_csv = Path("table/runs_manifest_20260421_141025_2.csv")
manifest_csv = Path("table/runs_manifest_20260506_011023 copy.csv")
evals_root = Path("evals")

# ---------------------------
# Table config
# ---------------------------
CONFIG_ORDER = ["CRW", "EE", "POI", "LPOI"]

MODEL_LABELS = {
    "rule-based": "Rule-based",
    "ppo": "PPO",
    "sac": "SAC",
    "dqn": "DQN",
}

MODEL_ORDER = ["rule-based", "dqn", "ppo", "sac"]
LEARNED_MODELS = ["dqn", "ppo", "sac"]

# (output_key, latex_label, source_column, transform_fn)
METRICS = [
    ("reward", "Total reward", "reward", lambda s: s),
    ("monitoring_reward", "Monitoring reward", "r_dist", lambda s: s + 1),
    ("disturbance_penalty", "Disturbance penalty", "p_disturbance", lambda s: s),
]

AGENT_CSVS = {
    "ppo": "ppo_ep.csv",
    "sac": "sac_ep.csv",
    "dqn": "dqn_ep.csv",
}

RULE_BASED_CSV = "CentroidStandoff_ep.csv"

# ---------------------------
# Helpers
# ---------------------------
def summarize_series(series: pd.Series) -> dict:
    mean = series.mean()
    std = series.std(ddof=1)
    return {
        "mean": mean,
        "std": std,
        "formatted": f"${mean:.3f} \\pm {std:.3f}$",
    }

def fmt_mean_only(x: float) -> str:
    return f"${x:.3f}$"

def read_eval_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, comment="#")
    required_cols = [source_col for _, _, source_col, _ in METRICS]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} is missing required columns: {missing}")
    return df

def summarize_df(df: pd.DataFrame) -> dict:
    return {
        output_key: summarize_series(transform_fn(df[source_col]))
        for output_key, _, source_col, transform_fn in METRICS
    }

# ---------------------------
# Load manifest
# ---------------------------
manifest = pd.read_csv(manifest_csv)

required_manifest_cols = {"config", "agent", "eval_name"}
missing_manifest_cols = required_manifest_cols - set(manifest.columns)
if missing_manifest_cols:
    raise ValueError(f"Manifest is missing required columns: {sorted(missing_manifest_cols)}")

# ---------------------------
# Aggregate results
# ---------------------------
results = {}

# PPO / SAC / DQN from agent-specific files
for _, row in manifest.iterrows():
    config = str(row["config"]).strip()
    agent = str(row["agent"]).strip().lower()
    eval_name = str(row["eval_name"]).strip()

    if agent not in AGENT_CSVS:
        raise ValueError(f"Unknown agent type: {agent}")

    csv_path = evals_root / eval_name / AGENT_CSVS[agent]
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing eval CSV for {agent}: {csv_path}")

    df = read_eval_csv(csv_path)

    results.setdefault(agent, {})
    results[agent][config] = summarize_df(df)

# Rule-based from CentroidStandoff_ep.csv
results["rule-based"] = {}

for config in CONFIG_ORDER:
    matching = manifest[manifest["config"].astype(str).str.strip() == config]
    if matching.empty:
        continue

    eval_name = str(matching.iloc[0]["eval_name"]).strip()
    csv_path = evals_root / eval_name / RULE_BASED_CSV

    if not csv_path.exists():
        raise FileNotFoundError(f"Missing rule-based CSV for {config}: {csv_path}")

    df = read_eval_csv(csv_path)
    results["rule-based"][config] = summarize_df(df)

# ---------------------------
# Compute learned-method averages
# ---------------------------
learned_avg = {}

for output_key, _, _, _ in METRICS:
    learned_avg[output_key] = {}

    for config in CONFIG_ORDER:
        vals = []
        for model_key in LEARNED_MODELS:
            if model_key in results and config in results[model_key]:
                vals.append(results[model_key][config][output_key]["mean"])

        learned_avg[output_key][config] = fmt_mean_only(sum(vals) / len(vals)) if vals else "--"

# ---------------------------
# Build LaTeX table
# ---------------------------
lines = []
lines.append(r"\begin{table}[!ht]")
lines.append(r"\centering")
lines.append(r"\setlength{\tabcolsep}{4pt}")
lines.append(
    r"\caption{Performance comparison across behaviors. Results are reported as mean $\pm$ standard deviation across evaluation runs of \(N=100\) repetitions. The final block reports the average mean across learned methods (DQN, PPO, and SAC).}"
)
lines.append(r"\label{tab:behavior_results_full}")
lines.append(r"\begin{tabular}{llcccc}")
lines.append(r"\hline \hline")
lines.append(r"\textbf{Model} & \textbf{Metric} & \textbf{CRW} & \textbf{EE} & \textbf{POI} & \textbf{LPOI} \\")
lines.append(r"\hline \hline")

models_present = [m for m in MODEL_ORDER if m in results and results[m]]

for mi, model_key in enumerate(models_present):
    model_block = results[model_key]
    model_label = MODEL_LABELS[model_key]

    for metric_idx, (output_key, metric_label, _, _) in enumerate(METRICS):
        row_cells = []
        row_cells.append(model_label if metric_idx == 0 else "")
        row_cells.append(metric_label)

        for config in CONFIG_ORDER:
            value = model_block.get(config, {}).get(output_key, {}).get("formatted", "--")
            row_cells.append(value)

        lines.append(" & ".join(row_cells) + r" \\")
        lines.append("")

    if mi < len(models_present) - 1:
        lines.append(r"\midrule")

# Final learned-average block
lines.append(r"\midrule")
for metric_idx, (output_key, metric_label, _, _) in enumerate(METRICS):
    row_cells = []
    row_cells.append(r"\textit{Avg. learned}" if metric_idx == 0 else "")
    row_cells.append(metric_label)

    for config in CONFIG_ORDER:
        row_cells.append(learned_avg[output_key][config])

    lines.append(" & ".join(row_cells) + r" \\")
    lines.append("")

lines.append(r"\hline \hline")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
print(latex_table)

\begin{table}[!ht]
\centering
\setlength{\tabcolsep}{4pt}
\caption{Performance comparison across behaviors. Results are reported as mean $\pm$ standard deviation across evaluation runs of \(N=100\) repetitions. The final block reports the average mean across learned methods (DQN, PPO, and SAC).}
\label{tab:behavior_results_full}
\begin{tabular}{llcccc}
\hline \hline
\textbf{Model} & \textbf{Metric} & \textbf{CRW} & \textbf{EE} & \textbf{POI} & \textbf{LPOI} \\
\hline \hline
Rule-based & Total reward & $0.772 \pm 0.024$ & $0.825 \pm 0.022$ & $0.771 \pm 0.022$ & $0.774 \pm 0.021$ \\

 & Monitoring reward & $0.485 \pm 0.006$ & $0.497 \pm 0.004$ & $0.483 \pm 0.007$ & $0.483 \pm 0.006$ \\

 & Disturbance penalty & $0.217 \pm 0.008$ & $0.198 \pm 0.008$ & $0.215 \pm 0.007$ & $0.214 \pm 0.007$ \\

\midrule
DQN & Total reward & $0.909 \pm 0.006$ & $0.886 \pm 0.009$ & $0.916 \pm 0.010$ & $0.829 \pm 0.020$ \\

 & Monitoring reward & $0.585 \pm 0.018$ & $0.546 \pm 0.013$ & $0.554 \pm 0.010$ & $0.4

In [3]:
from pathlib import Path
import re
import pandas as pd

# ---------------------------
# User-configurable paths
# ---------------------------
ROOT_CANDIDATES = [
    Path("evals/eval_animals_st_1/behaviors_t5")
]

# ---------------------------
# Table config
# ---------------------------
CONFIG_ORDER = ["CRW", "EE", "POI", "LPOI"]

MODEL_LABELS = {
    "rule-based": "Rule-based",
    "dqn": "DQN",
    "ppo": "PPO",
    "sac": "SAC",
}
MODEL_ORDER = ["rule-based", "dqn", "ppo", "sac"]

ANIMAL_DISPLAY = {
    "jackals_km_sm": "Jackals",
    "pigeons_km_sm": "Pigeons",
    "spur_winged_lapwings_km_sm": "Spur-winged lapwings",
    "spur_winged_lapwings": "Spur-winged lapwings",
}

# (output_key, latex_label, source_column, transform_fn)
METRICS = [
    ("reward", "Total reward", "reward", lambda s: s),
    ("monitoring_reward", "Monitoring reward", "r_dist", lambda s: s + 1),
    ("disturbance_penalty", "Disturbance penalty", "p_disturbance", lambda s: s),
]

AGENT_CSVS = {
    "ppo": "ppo_ep.csv",
    "sac": "sac_ep.csv",
    "dqn": "dqn_ep.csv",
}
RULE_BASED_CSV = "CentroidStandoff_ep.csv"

# ---------------------------
# Helpers
# ---------------------------
def resolve_root() -> Path:
    for root in ROOT_CANDIDATES:
        if root.exists():
            return root
    raise FileNotFoundError(
        "Could not find eval animal root. Tried:\n- "
        + "\n- ".join(str(p) for p in ROOT_CANDIDATES)
    )

def fmt_mean_std(series: pd.Series) -> str:
    mean = series.mean()
    std = series.std(ddof=1)
    return f"${mean:.3f} \\pm {std:.3f}$"

def read_eval_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, comment="#")
    required_cols = [source_col for _, _, source_col, _ in METRICS]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} is missing required columns: {missing}")
    return df

def summarize_df(df: pd.DataFrame) -> dict:
    return {
        output_key: fmt_mean_std(transform_fn(df[source_col]))
        for output_key, _, source_col, transform_fn in METRICS
    }

def selection_score(df: pd.DataFrame) -> float:
    # Best eval criterion: highest mean total reward
    return float(df["reward"].mean())

def animal_display_name(animal_key: str) -> str:
    return ANIMAL_DISPLAY.get(animal_key, animal_key.replace("_", " "))

def infer_config_from_path(path: Path) -> str | None:
    s = str(path).replace("\\", "/").upper()

    patterns = [
        ("LPOI", [r"(^|[^A-Z])LPOI([^A-Z]|$)"]),
        ("POI",  [r"(^|[^A-Z])POI([^A-Z]|$)"]),
        ("CRW",  [r"(^|[^A-Z])CRW([^A-Z]|$)"]),
        ("EE",   [r"(^|[^A-Z])EE([^A-Z]|$)"]),
    ]

    for cfg, pats in patterns:
        for pat in pats:
            if re.search(pat, s):
                return cfg
    return None

def find_animal_dirs(root: Path) -> dict[str, Path]:
    found = {}
    for child in root.iterdir():
        if not child.is_dir():
            continue
        name = child.name
        if "jackals" in name:
            found["jackals_km_sm"] = child
        elif "pigeons" in name:
            found["pigeons_km_sm"] = child
        elif "spur_winged_lapwings" in name:
            found[name] = child
    return found

def collect_csvs(animal_dir: Path, filename: str) -> list[Path]:
    return sorted(animal_dir.rglob(filename))

# ---------------------------
# Scan eval directories
# ---------------------------
root = resolve_root()
animal_dirs = find_animal_dirs(root)

if not animal_dirs:
    raise FileNotFoundError(f"No animal subdirectories found under {root}")

# full results for the big table
results = {}  # results[animal][model][config] = summary dict

# keep numeric scores so we can choose the best config later
scores = {}   # scores[animal][model][config] = mean reward score

for animal_key, animal_dir in animal_dirs.items():
    results.setdefault(animal_key, {})
    scores.setdefault(animal_key, {})

    # -----------------------
    # Agent-based models
    # -----------------------
    for agent, csv_name in AGENT_CSVS.items():
        best_by_config = {}

        for csv_path in collect_csvs(animal_dir, csv_name):
            config = infer_config_from_path(csv_path)
            if config not in CONFIG_ORDER:
                continue

            df = read_eval_csv(csv_path)
            score = selection_score(df)

            current = best_by_config.get(config)
            if current is None or score > current["score"]:
                best_by_config[config] = {
                    "score": score,
                    "summary": summarize_df(df),
                }

        if best_by_config:
            results[animal_key].setdefault(agent, {})
            scores[animal_key].setdefault(agent, {})
            for config, item in best_by_config.items():
                results[animal_key][agent][config] = item["summary"]
                scores[animal_key][agent][config] = item["score"]

    # -----------------------
    # Rule-based baseline
    # -----------------------
    baseline_by_config = {}
    baseline_scores = {}

    for csv_path in collect_csvs(animal_dir, RULE_BASED_CSV):
        config = infer_config_from_path(csv_path)
        if config not in CONFIG_ORDER:
            continue
        if config in baseline_by_config:
            continue

        df = read_eval_csv(csv_path)
        baseline_by_config[config] = summarize_df(df)
        baseline_scores[config] = selection_score(df)

    if baseline_by_config:
        results[animal_key]["rule-based"] = baseline_by_config
        scores[animal_key]["rule-based"] = baseline_scores

# ---------------------------
# Build full LaTeX table
# ---------------------------
full_lines = []
full_lines.append(r"\begin{table*}[!ht]")
full_lines.append(r"\centering")
full_lines.append(r"\setlength{\tabcolsep}{4pt}")
full_lines.append(
    r"\caption{Performance comparison across animals and behavior priors for each animal, movement type, and agent. "
    r"Results are reported as mean $\pm$ standard deviation across evaluation episodes.}"
)
full_lines.append(r"\label{tab:behavior_results_full_animals}")
full_lines.append(r"\begin{tabular}{lllcccc}")
full_lines.append(r"\hline \hline")
full_lines.append(
    r"\textbf{Animal} & \textbf{Model} & \textbf{Metric} & \textbf{CRW} & \textbf{EE} & \textbf{POI} & \textbf{LPOI} \\"
)
full_lines.append(r"\hline \hline")

animal_keys_sorted = sorted(results.keys(), key=lambda k: animal_display_name(k).lower())

first_animal = True
for animal_key in animal_keys_sorted:
    animal_block = results[animal_key]
    animal_label = animal_display_name(animal_key)
    models_present = [m for m in MODEL_ORDER if m in animal_block and animal_block[m]]

    if not models_present:
        continue

    if not first_animal:
        full_lines.append(r"\midrule")
    first_animal = False

    animal_row_count = len(models_present) * len(METRICS)
    animal_row_idx = 0

    for mi, model_key in enumerate(models_present):
        model_block = animal_block[model_key]
        model_label = MODEL_LABELS[model_key]

        for metric_idx, (output_key, metric_label, _, _) in enumerate(METRICS):
            row_cells = []

            if animal_row_idx == 0:
                row_cells.append(rf"\multirow{{{animal_row_count}}}{{*}}{{{animal_label}}}")
            else:
                row_cells.append("")

            row_cells.append(model_label if metric_idx == 0 else "")
            row_cells.append(metric_label)

            for config in CONFIG_ORDER:
                value = model_block.get(config, {}).get(output_key, "--")
                row_cells.append(value)

            full_lines.append(" & ".join(row_cells) + r" \\")
            animal_row_idx += 1

        if mi < len(models_present) - 1:
            full_lines.append(r"\cmidrule(lr){2-7}")

full_lines.append(r"\hline \hline")
full_lines.append(r"\end{tabular}")
full_lines.append(r"\end{table*}")

# ---------------------------
# Build selected table
# best config among CRW / EE / POI / LPOI
# animals on columns
# ---------------------------
best_choice = {}  # best_choice[model][animal] = {"config": ..., "summary": ...}

for animal_key in animal_keys_sorted:
    best_choice.setdefault(animal_key, {})
    animal_scores = scores.get(animal_key, {})
    animal_results = results.get(animal_key, {})

    for model_key in MODEL_ORDER:
        model_scores = animal_scores.get(model_key, {})
        if not model_scores:
            continue

        best_config = max(model_scores, key=model_scores.get)
        best_choice[animal_key][model_key] = {
            "config": best_config,
            "summary": animal_results[model_key][best_config],
        }

selected_lines = []
selected_lines.append(r"\begin{table*}[!ht]")
selected_lines.append(r"\centering")
selected_lines.append(r"\setlength{\tabcolsep}{4pt}")
selected_lines.append(
    r"\caption{Performance comparison across animals and behavior priors for each animal, and agent using the best performing movement type. "
    r"Results are reported as mean $\pm$ standard deviation across evaluation episodes.}"
)
selected_lines.append(r"\label{tab:behavior_results_selected_animals}")
selected_lines.append(r"\begin{tabular}{llccc}")
selected_lines.append(r"\hline \hline")
selected_lines.append(
    r"\textbf{Model} & \textbf{Metric} & \textbf{Jackals} & \textbf{Pigeons} & \textbf{Spur-winged lapwings} \\"
)
selected_lines.append(r"\hline \hline")

animal_col_order = []
for key in animal_keys_sorted:
    label = animal_display_name(key)
    if label == "Jackals":
        animal_col_order.append(key)
for key in animal_keys_sorted:
    label = animal_display_name(key)
    if label == "Pigeons":
        animal_col_order.append(key)
for key in animal_keys_sorted:
    label = animal_display_name(key)
    if label == "Spur-winged lapwings":
        animal_col_order.append(key)

animal_col_order = list(dict.fromkeys(animal_col_order))

models_present = []
for model_key in MODEL_ORDER:
    present = any(model_key in best_choice.get(animal_key, {}) for animal_key in animal_col_order)
    if present:
        models_present.append(model_key)

for mi, model_key in enumerate(models_present):
    model_label = MODEL_LABELS[model_key]

    for metric_idx, (output_key, metric_label, _, _) in enumerate(METRICS):
        row_cells = []
        row_cells.append(model_label if metric_idx == 0 else "")
        row_cells.append(metric_label)

        for animal_key in animal_col_order:
            item = best_choice.get(animal_key, {}).get(model_key)
            if item is None:
                row_cells.append("--")
            else:
                cfg = item["config"]
                value = item["summary"][output_key]
                row_cells.append(rf"{value} ({cfg})")

        selected_lines.append(" & ".join(row_cells) + r" \\")
        selected_lines.append("")

    if mi < len(models_present) - 1:
        selected_lines.append(r"\midrule")

selected_lines.append(r"\hline \hline")
selected_lines.append(r"\end{tabular}")
selected_lines.append(r"\end{table*}")

# ---------------------------
# Print both tables
# ---------------------------
print("\n".join(full_lines))
print()
print("\n".join(selected_lines))

\begin{table*}[!ht]
\centering
\setlength{\tabcolsep}{4pt}
\caption{Performance comparison across animals and behavior priors for each animal, movement type, and agent. Results are reported as mean $\pm$ standard deviation across evaluation episodes.}
\label{tab:behavior_results_full_animals}
\begin{tabular}{lllcccc}
\hline \hline
\textbf{Animal} & \textbf{Model} & \textbf{Metric} & \textbf{CRW} & \textbf{EE} & \textbf{POI} & \textbf{LPOI} \\
\hline \hline
\multirow{9}{*}{Jackals} & DQN & Total reward & $0.917 \pm 0.010$ & $0.929 \pm 0.004$ & $0.887 \pm 0.008$ & $0.919 \pm 0.007$ \\
 &  & Monitoring reward & $0.568 \pm 0.029$ & $0.564 \pm 0.006$ & $0.616 \pm 0.023$ & $0.568 \pm 0.012$ \\
 &  & Disturbance penalty & $0.198 \pm 0.023$ & $0.188 \pm 0.005$ & $0.252 \pm 0.021$ & $0.191 \pm 0.010$ \\
\cmidrule(lr){2-7}
 & PPO & Total reward & $0.898 \pm 0.012$ & $0.895 \pm 0.010$ & $0.910 \pm 0.010$ & $0.916 \pm 0.008$ \\
 &  & Monitoring reward & $0.560 \pm 0.044$ & $0.504 \pm 0.022$ & $0.5

In [7]:
from pathlib import Path
import pandas as pd

# manifest_csv = Path("table/replay_eval_manifest_20260505_164832.csv")
# evals_root = Path("evals/replay1")

manifest_csv = Path("table/replay_eval_manifest_20260505_225338.csv")
evals_root = Path("evals/replay2")

SPECIES_ORDER = ["JACKALS", "PIGEONS", "SPUR"]
TRAINING_ORDER = ["BEST", "GPS"]
MODEL_ORDER = ["sac"]

SPECIES_LABELS = {
    "JACKALS": "Jackal",
    "PIGEONS": "Pigeon",
    "SPUR": "Spur-winged lapwing",
}

MODEL_LABELS = {
    "sac": "SAC",
}

BEST_TRAINING_LABELS = {
    "JACKALS": "LPOI",
    "PIGEONS": "LPOI",
    "SPUR": "EE",
}

# (output_key, row_label, source_column, transform)
METRICS = [
    ("reward", "Total reward", "reward", lambda s: s),
    ("monitoring_reward", "Monitoring reward", "r_dist", lambda s: s + 1),
    ("disturbance_penalty", "Disturbance penalty", "p_disturbance", lambda s: s),
]

def fmt_mean_std(series: pd.Series) -> str:
    return f"${series.mean():.3f} \\pm {series.std(ddof=1):.3f}$"

manifest = pd.read_csv(manifest_csv)

results = {}

for _, row in manifest.iterrows():
    cfg = str(row["config"]).strip()
    agent = str(row["agent"]).strip().lower()
    eval_name = str(row["eval_name"]).strip()

    if cfg.startswith("JACKALS"):
        species = "JACKALS"
    elif cfg.startswith("PIGEONS"):
        species = "PIGEONS"
    elif cfg.startswith("SPUR"):
        species = "SPUR"
    else:
        continue

    training_data = "GPS" if "GPS" in cfg else "BEST"

    ep_csv = evals_root / eval_name / f"{agent}_ep.csv"
    if not ep_csv.exists():
        raise FileNotFoundError(ep_csv)

    df = pd.read_csv(ep_csv, comment="#")

    results.setdefault(training_data, {})
    results[training_data].setdefault(agent, {})
    results[training_data][agent][species] = {
        out_key: fmt_mean_std(transform(df[source_col]))
        for out_key, _, source_col, transform in METRICS
    }

lines = []
lines.append(r"\begin{table}[!ht]")
lines.append(r"\centering")
lines.append(r"\setlength{\tabcolsep}{4pt}")
lines.append(r"\caption{Replay evaluation performance across species, training data, and models. Results are reported as mean $\pm$ standard deviation across 100 evaluation episodes.}")
lines.append(r"\label{tab:replay_results_all_species}")
lines.append(r"\begin{tabular}{llcccc}")
lines.append(r"\hline \hline")
lines.append(r"\textbf{Training} & \textbf{Model} & \textbf{Metric} & \textbf{Jackal} & \textbf{Pigeon} & \textbf{Spur-winged lapwing} \\")
lines.append(r"\hline \hline")

row_blocks = []
for training_data in TRAINING_ORDER:
    training_label = {
        "JACKALS": "GPS" if training_data == "GPS" else BEST_TRAINING_LABELS["JACKALS"],
        "PIGEONS": "GPS" if training_data == "GPS" else BEST_TRAINING_LABELS["PIGEONS"],
        "SPUR": "GPS" if training_data == "GPS" else BEST_TRAINING_LABELS["SPUR"],
    }

    # single label for the row block
    block_label = "GPS" if training_data == "GPS" else "Best-fit"

    for agent in MODEL_ORDER:
        for metric_idx, (out_key, metric_label, _, _) in enumerate(METRICS):
            training_cell = rf"\multirow{{6}}{{*}}{{{block_label}}}" if (agent == MODEL_ORDER[0] and metric_idx == 0) else ""
            model_cell = rf"\multirow{{3}}{{*}}{{{MODEL_LABELS[agent]}}}" if metric_idx == 0 else ""

            row = [
                training_cell,
                model_cell,
                metric_label,
            ]

            for species in SPECIES_ORDER:
                value = results.get(training_data, {}).get(agent, {}).get(species, {}).get(out_key, "--")
                row.append(value)

            lines.append(" & ".join(row) + r" \\")
        lines.append("")

    if training_data != TRAINING_ORDER[-1]:
        lines.append(r"\midrule")

lines.append(r"\hline \hline")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
print(latex_table)

\begin{table}[!ht]
\centering
\setlength{\tabcolsep}{4pt}
\caption{Replay evaluation performance across species, training data, and models. Results are reported as mean $\pm$ standard deviation across 100 evaluation episodes.}
\label{tab:replay_results_all_species}
\begin{tabular}{llcccc}
\hline \hline
\textbf{Training} & \textbf{Model} & \textbf{Metric} & \textbf{Jackal} & \textbf{Pigeon} & \textbf{Spur-winged lapwing} \\
\hline \hline
\multirow{6}{*}{Best-fit} & \multirow{3}{*}{SAC} & Total reward & $0.930 \pm 0.006$ & $0.876 \pm 0.051$ & $0.930 \pm 0.010$ \\
 &  & Monitoring reward & $0.582 \pm 0.011$ & $0.523 \pm 0.021$ & $0.568 \pm 0.006$ \\
 &  & Disturbance penalty & $0.200 \pm 0.012$ & $0.175 \pm 0.014$ & $0.185 \pm 0.009$ \\

\midrule
\multirow{6}{*}{GPS} & \multirow{3}{*}{SAC} & Total reward & $0.931 \pm 0.008$ & $0.860 \pm 0.128$ & $0.891 \pm 0.140$ \\
 &  & Monitoring reward & $0.565 \pm 0.009$ & $0.497 \pm 0.021$ & $0.525 \pm 0.025$ \\
 &  & Disturbance penalty & $0.185 \p

In [19]:
from pathlib import Path

import numpy as np
import pandas as pd
from pyproj import Transformer
from pyproj.aoi import AreaOfInterest
from pyproj.database import query_utm_crs_info


DATASETS = {
    "Jackals": {
        "input_path": "data/jackals/jackal_data.csv",
        "kind": "single_csv",
        "usecols": ["lat", "lon", "TAG", "dateTime_local"],
        "lat_col": "lat",
        "lon_col": "lon",
        "id_col": "TAG",
        "time_col": "dateTime_local",
        "t_col": None,
    },
    "Pigeons": {
        "input_path": "data/pigeons",
        "kind": "csv_directory",
        "usecols": ["lat", "lon", "t"],
        "lat_col": "lat",
        "lon_col": "lon",
        "id_col": "file_id",
        "time_col": None,
        "t_col": "t",
    },
    "Spur-winged lapwings": {
        "input_path": "data/spur_winged_lapwings/spur_winged_lapwings1.csv",
        "kind": "single_csv",
        "usecols": ["location-lat", "location-long", "tag-local-identifier", "timestamp"],
        "lat_col": "location-lat",
        "lon_col": "location-long",
        "id_col": "tag-local-identifier",
        "time_col": "timestamp",
        "t_col": None,
    },
}


def pick_utm_epsg(s_lat, n_lat, w_lon, e_lon, datum_name="WGS 84"):
    utm_list = query_utm_crs_info(
        datum_name=datum_name,
        area_of_interest=AreaOfInterest(
            west_lon_degree=w_lon,
            south_lat_degree=s_lat,
            east_lon_degree=e_lon,
            north_lat_degree=n_lat,
        ),
    )

    if not utm_list:
        raise ValueError(
            f"No UTM CRS found for lat=[{s_lat}, {n_lat}], lon=[{w_lon}, {e_lon}]"
        )

    return f"{utm_list[0].auth_name}:{utm_list[0].code}"


def transform_df_to_utm(df, lat_col, lon_col):
    utm_epsg = pick_utm_epsg(
        s_lat=df[lat_col].min(),
        n_lat=df[lat_col].max(),
        w_lon=df[lon_col].min(),
        e_lon=df[lon_col].max(),
    )

    transformer = Transformer.from_crs("EPSG:4326", utm_epsg, always_xy=True)

    x, y = transformer.transform(
        df[lon_col].to_numpy(),
        df[lat_col].to_numpy(),
    )

    out = df.copy()
    out["x"] = x
    out["y"] = y
    out["utm_epsg"] = utm_epsg

    return out


def read_dataset(cfg):
    path = Path(cfg["input_path"])

    if cfg["kind"] == "single_csv":
        df = pd.read_csv(
            path,
            usecols=cfg["usecols"],
            na_values=["NA"],
        )

    elif cfg["kind"] == "csv_directory":
        dfs = []

        for csv_path in sorted(path.glob("*.csv")):
            g = pd.read_csv(
                csv_path,
                usecols=cfg["usecols"],
                na_values=["NA"],
            )

            # For pigeons, each CSV file is treated as one animal/tag.
            g["file_id"] = csv_path.stem
            dfs.append(g)

        if not dfs:
            raise FileNotFoundError(f"No CSV files found in {path}")

        df = pd.concat(dfs, ignore_index=True)

    else:
        raise ValueError(f"Unknown dataset kind: {cfg['kind']}")

    return df


def add_time_and_movement_metrics(df, cfg):
    out = df.copy()

    lat_col = cfg["lat_col"]
    lon_col = cfg["lon_col"]
    id_col = cfg["id_col"]
    time_col = cfg["time_col"]
    t_col = cfg["t_col"]

    out[lat_col] = pd.to_numeric(out[lat_col], errors="coerce")
    out[lon_col] = pd.to_numeric(out[lon_col], errors="coerce")

    if time_col is not None:
        out[time_col] = pd.to_datetime(out[time_col], errors="coerce")
        out = out.sort_values([id_col, time_col], kind="stable")
        out["dt_seconds"] = out.groupby(id_col)[time_col].diff().dt.total_seconds()

    elif t_col is not None:
        out[t_col] = pd.to_numeric(out[t_col], errors="coerce")
        out = out.sort_values([id_col, t_col], kind="stable")
        out["dt_seconds"] = out.groupby(id_col)[t_col].diff()

    else:
        raise ValueError("Either time_col or t_col must be defined.")

    out = transform_df_to_utm(out, lat_col=lat_col, lon_col=lon_col)

    out["dx"] = out.groupby(id_col)["x"].diff()
    out["dy"] = out.groupby(id_col)["y"].diff()
    out["step_dist_m"] = np.hypot(out["dx"], out["dy"])

    out["speed_mps"] = out["step_dist_m"] / out["dt_seconds"]

    bad_dt = out["dt_seconds"].isna() | (out["dt_seconds"] <= 0)
    out.loc[bad_dt, ["step_dist_m", "speed_mps"]] = np.nan

    out["speed_mps"] = out["speed_mps"].replace([np.inf, -np.inf], np.nan)

    return out


def fmt_large(x):
    """Compact formatting for large counts."""
    if pd.isna(x):
        return "--"

    x = float(x)

    if abs(x) >= 1_000_000:
        return f"{x / 1_000_000:.2f}M"
    elif abs(x) >= 1_000:
        return f"{x / 1_000:.1f}k"
    else:
        return f"{x:.0f}"


def summarize_dataset(name, cfg, gap_threshold_s=20):
    df = read_dataset(cfg)
    df = add_time_and_movement_metrics(df, cfg)

    id_col = cfg["id_col"]
    time_col = cfg["time_col"]
    t_col = cfg["t_col"]

    grouped = df.groupby(id_col, sort=False)

    n_animals = df[id_col].nunique()
    n_samples = len(df)

    n_gaps = int((df["dt_seconds"] > gap_threshold_s).sum())

    x_extent = df["x"].max() - df["x"].min()
    y_extent = df["y"].max() - df["y"].min()

    mean_samples_per_animal = grouped.size().mean()

    # Mean dt within each animal, then averaged across animals.
    mean_dt_per_animal_s = grouped["dt_seconds"].apply(
        lambda s: s.loc[s > 0].mean()
    )
    mean_dt_within_animal_s = mean_dt_per_animal_s.mean()

    if time_col is not None:
        start_date = df[time_col].min().date()
        end_date = df[time_col].max().date()

        tracking_time_per_animal_s = (
            grouped[time_col].max() - grouped[time_col].min()
        ).dt.total_seconds()

    else:
        # Pigeon data uses relative time t rather than calendar timestamps.
        start_date = "--"
        end_date = "--"

        tracking_time_per_animal_s = grouped[t_col].max() - grouped[t_col].min()

    mean_tracking_time_h = tracking_time_per_animal_s.mean() / 3600

    return {
        "Species": name,
        "N": int(n_animals),
        "N_gap": n_gaps,
        "x": round(float(x_extent)),
        "y": round(float(y_extent)),
        "Start": start_date,
        "End": end_date,
        "T_bar": round(float(mean_tracking_time_h), 1),
        "dt_bar": round(float(mean_dt_within_animal_s), 2),
        "n_bar": round(float(mean_samples_per_animal)),
        "n": int(n_samples),
    }


rows = []

for name, cfg in DATASETS.items():
    print(f"Summarising {name}...")
    rows.append(summarize_dataset(name, cfg))

table = pd.DataFrame(rows)

# Notebook display table
table

latex_table_df = table.copy()

latex_table_df = latex_table_df.rename(columns={
    "N": r"$N$",
    "N_gap": r"$N_{\mathrm{gap}}$",
    "x": r"$x$",
    "y": r"$y$",
    "T_bar": r"$\bar{T}$",
    "dt_bar": r"$\bar{\Delta t}$",
    "n_bar": r"$\bar{n}$",
    "n": r"$n$",
})

# Compact large-count columns for the thesis table.
latex_table_df[r"$N_{\mathrm{gap}}$"] = latex_table_df[r"$N_{\mathrm{gap}}$"].apply(fmt_large)
latex_table_df[r"$\bar{n}$"] = latex_table_df[r"$\bar{n}$"].apply(fmt_large)
latex_table_df[r"$n$"] = latex_table_df[r"$n$"].apply(fmt_large)

# Format numeric columns.
latex_table_df[r"$x$"] = latex_table_df[r"$x$"].map(lambda v: f"{v:.0f}")
latex_table_df[r"$y$"] = latex_table_df[r"$y$"].map(lambda v: f"{v:.0f}")
latex_table_df[r"$\bar{T}$"] = latex_table_df[r"$\bar{T}$"].map(lambda v: f"{v:.1f}")
latex_table_df[r"$\bar{\Delta t}$"] = latex_table_df[r"$\bar{\Delta t}$"].map(lambda v: f"{v:.2f}")

latex_tabular = latex_table_df.to_latex(
    index=False,
    escape=False,
    column_format="@{}lrrrrllrrrr@{}",
)

latex_tabular = latex_tabular.replace(
    r"\begin{tabular}",
    r"\setlength{\tabcolsep}{3pt}" + "\n" + r"\begin{tabular}"
)

latex_table = (
r"""\begin{table}[htbp]
\centering
\small
\caption{
Summary of the movement datasets.
$N$ is the number of animals, or files for pigeons.
$N_{\mathrm{gap}}$ is the number of sampling gaps greater than 20 s.
$x$ and $y$ are spatial extents in metres.
$\bar{T}$ is mean tracking time per animal in hours.
$\bar{\Delta t}$ is the mean within-animal sampling interval in seconds.
$\bar{n}$ is the mean number of samples per animal, and $n$ is the total number of samples.
}
\label{tab:data_summary}
"""
+ latex_tabular +
r"""\end{table}"""
)

print(latex_table)

Summarising Jackals...
Summarising Pigeons...
Summarising Spur-winged lapwings...
\begin{table}[htbp]
\centering
\small
\caption{
Summary of the movement datasets.
$N$ is the number of animals, or files for pigeons.
$N_{\mathrm{gap}}$ is the number of sampling gaps greater than 20 s.
$x$ and $y$ are spatial extents in metres.
$\bar{T}$ is mean tracking time per animal in hours.
$\bar{\Delta t}$ is the mean within-animal sampling interval in seconds.
$\bar{n}$ is the mean number of samples per animal, and $n$ is the total number of samples.
}
\label{tab:data_summary}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{@{}lrrrrllrrrr@{}}
\toprule
Species & $N$ & $N_{\mathrm{gap}}$ & $x$ & $y$ & Start & End & $\bar{T}$ & $\bar{\Delta t}$ & $\bar{n}$ & $n$ \\
\midrule
Jackals & 50 & 233.7k & 13520 & 12475 & 2022-02-25 & 2025-02-18 & 2140.9 & 1422.74 & 139.4k & 6.97M \\
Pigeons & 17 & 0 & 917 & 2433 & -- & -- & 9.8 & 5.00 & 7.1k & 120.2k \\
Spur-winged lapwings & 16 & 1.39M & 17465 & 7724 & 2022-04

In [20]:
from pathlib import Path

import numpy as np
import pandas as pd


FILTERED_DATASETS = {
    "Jackals": {
        "full_dir": "track_segments/jackals/full",
        "episodes_dir": "track_segments/jackals/episodes/all",
        "id_col": "TAG",
        "has_dates": True,
    },
    "Pigeons": {
        "full_dir": "track_segments/pigeons/full",
        "episodes_dir": "track_segments/pigeons/episodes/all",
        "id_col": "file_id",
        "has_dates": False,
    },
    "Spur-winged lapwings": {
        "full_dir": "track_segments/spur_winged_lapwings/full",
        "episodes_dir": "track_segments/spur_winged_lapwings/episodes/all",
        "id_col": "tag-local-identifier",
        "has_dates": True,
    },
}


def fmt_large(x):
    """Compact formatting for large counts."""
    if pd.isna(x):
        return "--"

    x = float(x)

    if abs(x) >= 1_000_000:
        return f"{x / 1_000_000:.2f}M"
    elif abs(x) >= 1_000:
        return f"{x / 1_000:.1f}k"
    else:
        return f"{x:.0f}"


def fmt_mean_sd(mean, sd, decimals=2):
    if pd.isna(mean):
        return "--"
    if pd.isna(sd):
        sd = 0.0
    return f"{mean:.{decimals}f} $\\pm$ {sd:.{decimals}f}"


def read_manifest(path):
    path = Path(path)

    if path.suffix == ".parquet":
        return pd.read_parquet(path)

    if path.suffix == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Unsupported manifest format: {path}")


def segment_step_stats(npz_path, gap_threshold_s=20):
    """
    Compute dt, speed, and gap statistics from one saved filtered segment.
    The .npz files are expected to contain t, x, and y arrays.
    """
    z = np.load(npz_path)

    t = z["t"].astype(float)
    x = z["x"].astype(float)
    y = z["y"].astype(float)

    if len(t) < 2:
        return {
            "n_dt": 0,
            "dt_sum": 0.0,
            "n_gaps": 0,
            "n_speed": 0,
            "speed_sum": 0.0,
            "speed_sumsq": 0.0,
        }

    dt = np.diff(t)
    dx = np.diff(x)
    dy = np.diff(y)

    step_dist = np.hypot(dx, dy)

    valid_dt = np.isfinite(dt) & (dt > 0)
    speed = np.full_like(dt, np.nan, dtype=float)
    speed[valid_dt] = step_dist[valid_dt] / dt[valid_dt]

    valid_speed = np.isfinite(speed)

    return {
        "n_dt": int(valid_dt.sum()),
        "dt_sum": float(dt[valid_dt].sum()),
        "n_gaps": int((dt > gap_threshold_s).sum()),
        "n_speed": int(valid_speed.sum()),
        "speed_sum": float(speed[valid_speed].sum()),
        "speed_sumsq": float(np.square(speed[valid_speed]).sum()),
    }


def summarize_filtered_dataset(name, cfg, gap_threshold_s=20):
    full_dir = Path(cfg["full_dir"])
    episodes_dir = Path(cfg["episodes_dir"])
    id_col = cfg["id_col"]

    manifest_path = full_dir / "manifest.parquet"
    manifest = read_manifest(manifest_path)

    if id_col not in manifest.columns:
        raise ValueError(
            f"Expected id column '{id_col}' in {manifest_path}, "
            f"but found columns: {list(manifest.columns)}"
        )

    n_animals = manifest[id_col].nunique()
    n_segments = len(manifest)
    n_samples = int(manifest["n_points"].sum())

    if episodes_dir.exists() and (episodes_dir / "manifest.parquet").exists():
        episode_manifest = read_manifest(episodes_dir / "manifest.parquet")
        n_episodes = len(episode_manifest)
    else:
        n_episodes = np.nan

    x_extent = manifest["xmax"].max() - manifest["xmin"].min()
    y_extent = manifest["ymax"].max() - manifest["ymin"].min()

    samples_per_animal = manifest.groupby(id_col)["n_points"].sum()
    mean_samples_per_animal = samples_per_animal.mean()

    # For filtered data, mean tracking time per animal is the sum of retained
    # filtered segment durations per animal, not first-to-last calendar span.
    tracking_time_per_animal_s = manifest.groupby(id_col)["duration_s"].sum()
    mean_tracking_time_h = tracking_time_per_animal_s.mean() / 3600

    if cfg["has_dates"] and "start_time" in manifest.columns and "end_time" in manifest.columns:
        start_date = pd.to_datetime(manifest["start_time"], errors="coerce").min().date()
        end_date = pd.to_datetime(manifest["end_time"], errors="coerce").max().date()
    else:
        start_date = "--"
        end_date = "--"

    # Step-level dt and speed stats are computed from saved .npz files.
    animal_dt_sum = {}
    animal_dt_n = {}

    speed_sum = 0.0
    speed_sumsq = 0.0
    speed_n = 0

    n_gaps = 0

    for _, row in manifest.iterrows():
        animal_id = row[id_col]
        npz_path = full_dir / row["path"]

        stats = segment_step_stats(npz_path, gap_threshold_s=gap_threshold_s)

        animal_dt_sum[animal_id] = animal_dt_sum.get(animal_id, 0.0) + stats["dt_sum"]
        animal_dt_n[animal_id] = animal_dt_n.get(animal_id, 0) + stats["n_dt"]

        speed_sum += stats["speed_sum"]
        speed_sumsq += stats["speed_sumsq"]
        speed_n += stats["n_speed"]

        n_gaps += stats["n_gaps"]

    mean_dt_per_animal = []

    for animal_id in animal_dt_sum:
        if animal_dt_n[animal_id] > 0:
            mean_dt_per_animal.append(animal_dt_sum[animal_id] / animal_dt_n[animal_id])

    mean_dt_within_animal_s = float(np.mean(mean_dt_per_animal)) if mean_dt_per_animal else np.nan

    if speed_n > 0:
        speed_mean = speed_sum / speed_n

        if speed_n > 1:
            speed_var = (speed_sumsq - (speed_sum ** 2) / speed_n) / (speed_n - 1)
            speed_sd = np.sqrt(max(speed_var, 0.0))
        else:
            speed_sd = 0.0
    else:
        speed_mean = np.nan
        speed_sd = np.nan

    return {
        "Species": name,
        "N": int(n_animals),
        "S": int(n_segments),
        "E": int(n_episodes) if not pd.isna(n_episodes) else np.nan,
        "Speed": fmt_mean_sd(speed_mean, speed_sd),
        "N_gap": int(n_gaps),
        "x": round(float(x_extent)),
        "y": round(float(y_extent)),
        "Start": start_date,
        "End": end_date,
        "T_bar": round(float(mean_tracking_time_h), 1),
        "dt_bar": round(float(mean_dt_within_animal_s), 2),
        "n_bar": round(float(mean_samples_per_animal)),
        "n": int(n_samples),
    }


rows = []

for name, cfg in FILTERED_DATASETS.items():
    print(f"Summarising filtered dataset: {name}...")
    rows.append(summarize_filtered_dataset(name, cfg))

filtered_table = pd.DataFrame(rows)

display(filtered_table)

Summarising filtered dataset: Jackals...
Summarising filtered dataset: Pigeons...
Summarising filtered dataset: Spur-winged lapwings...


,Species,N,S,E,Speed,N_gap,x,y,Start,End,T_bar,dt_bar,n_bar,n
0,Jackals,45,5096,12168,5.22 $\pm$ 3.51,0,12494,9780,2022-02-25,2025-02-18,18.4,2.02,34566,1555461
1,Pigeons,17,17,2925,5.81 $\pm$ 7.68,0,917,2433,--,--,9.8,5.01,7059,119996
2,Spur-winged lapwings,16,2158,16602,0.75 $\pm$ 1.84,0,9696,6886,2022-04-24,2023-03-25,63.7,8.24,28027,448426


In [ ]:
latex_table_df = filtered_table.copy()

# Drop columns not needed in the filtered-data table.
latex_table_df = latex_table_df.drop(
    columns=["Start", "End", "N_gap"],
    errors="ignore"
)

latex_table_df = latex_table_df.rename(columns={
    "N": r"$N$",
    "S": r"$S$",
    "E": r"$E$",
    "Speed": r"$v$",
    "x": r"$x$",
    "y": r"$y$",
    "T_bar": r"$\bar{T}$",
    "dt_bar": r"$\bar{\Delta t}$",
    "n_bar": r"$\bar{n}$",
    "n": r"$n$",
})

latex_table_df[r"$\bar{n}$"] = latex_table_df[r"$\bar{n}$"].apply(fmt_large)
latex_table_df[r"$n$"] = latex_table_df[r"$n$"].apply(fmt_large)
latex_table_df[r"$S$"] = latex_table_df[r"$S$"].apply(fmt_large)
latex_table_df[r"$E$"] = latex_table_df[r"$E$"].apply(fmt_large)

latex_table_df[r"$x$"] = latex_table_df[r"$x$"].map(lambda v: f"{v:.0f}")
latex_table_df[r"$y$"] = latex_table_df[r"$y$"].map(lambda v: f"{v:.0f}")
latex_table_df[r"$\bar{T}$"] = latex_table_df[r"$\bar{T}$"].map(lambda v: f"{v:.1f}")
latex_table_df[r"$\bar{\Delta t}$"] = latex_table_df[r"$\bar{\Delta t}$"].map(lambda v: f"{v:.2f}")

latex_tabular = latex_table_df.to_latex(
    index=False,
    escape=False,
    column_format="@{}lrrrlrrrrrr@{}",
)

latex_tabular = latex_tabular.replace(
    r"\begin{tabular}",
    r"\setlength{\tabcolsep}{3pt}" + "\n" + r"\begin{tabular}"
)

latex_table = (
r"""\begin{table}[htbp]
\centering
\small
\caption{
Summary of the filtered movement datasets.
$N$ is the number of animals.
$S$ is the number of contigous trajectory segments and $E$ is the number of episode duration segments.
$v$ is mean step speed in m s$^{-1}$, shown as mean $\pm$ standard deviation.
$x$ and $y$ are spatial extents in metres.
$\bar{T}$ is the mean retained tracking time per animal in hours.
$\bar{\Delta t}$ is the mean within-animal sampling interval in seconds.
$\bar{n}$ is the mean number of retained samples per animal, and $n$ is the total number of retained samples.
}
\label{tab:filtered_data_summary}
"""
+ latex_tabular +
r"""\end{table}"""
)

print(latex_table)

\begin{table}[htbp]
\centering
\small
\caption{
Summary of the filtered movement datasets.
$N$ is the number of animals.
$S$ is the number of contigous trajectory segments and $E$ is the number of episode duration segments.
$v$ is mean step speed in m s$^{-1}$, shown as mean $\pm$ standard deviation.
$x$ and $y$ are spatial extents in metres.
$\bar{T}$ is the mean retained tracking time per animal in hours.
$\bar{\Delta t}$ is the mean within-animal sampling interval in seconds.
$\bar{n}$ is the mean number of retained samples per animal, and $n$ is the total number of retained samples.
}
\label{tab:filtered_data_summary}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{@{}lrrrlrrrrr@{}}
\toprule
Species & $N$ & $S$ & $E$ & $v$ & $x$ & $y$ & $\bar{T}$ & $\bar{\Delta t}$ & $\bar{n}$ & $n$ \\
\midrule
Jackals & 45 & 5.1k & 12.2k & 5.22 $\pm$ 3.51 & 12494 & 9780 & 18.4 & 2.02 & 34.6k & 1.56M \\
Pigeons & 17 & 17 & 2.9k & 5.81 $\pm$ 7.68 & 917 & 2433 & 9.8 & 5.01 & 7.1k & 120.0k \\
Spur-winged 